# 05 — t-SNE of Pruned Models in Metric Spaces

Each **point** represents one model variant (calibration source × pruning level).
There are 13 points per plot: 3 calibration schemas × 4 pruning levels (20/40/60/80%) + 1 shared baseline.

The feature vector for each model is the **N-dimensional vector of per-example metric values**
computed over a given eval benchmark (N=1319 for GSM8K, N=33 for HumanEval+, N=1171 for ARC-Challenge).
The baseline has an all-zero feature vector (metric of a distribution vs. itself = 0).

**Layout**: 4 figures (one per metric: KLD, JSD, EMD, Chamfer), each with 3 panels (one per eval benchmark).

**Goal**: do models cluster by pruning level (how much was pruned) or by calibration source (what was
used to determine what to prune)?  If pruning level dominates, points should form concentric shells
around the baseline.  If calibration source dominates, the three cal groups should be spatially separate.

In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /var/home/three-kingdoms/work/pruning-metrics


In [2]:
import json
import re
import concurrent.futures
import boto3

AWS_PROFILE    = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get("RESULTS_BUCKET", "pruning-metrics-results-414266451290")

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

experiment_config_path = NOTEBOOK_DIR / "experiment_config.json"
_cfg = json.loads(experiment_config_path.read_text(encoding="utf-8"))

TF_URIS = _cfg["teacher_forced_uris"]

PRUNING_LEVELS  = [0, 20, 40, 60, 80]
NONZERO_LEVELS  = [20, 40, 60, 80]
DATASETS        = ["arc_challenge", "gsm8k", "humaneval"]
BENCHMARK_LABELS = {
    "gsm8k":         "GSM8K",
    "humaneval":     "HumanEval+",
    "arc_challenge": "ARC-Challenge",
}

TF_CACHE_DIR = RESULTS_DIR / "tf_cache"
TF_CACHE_DIR.mkdir(exist_ok=True)

session = boto3.session.Session(profile_name=AWS_PROFILE)
s3      = session.client("s3")


def _parse_combo(combo_key: str) -> tuple[str, str]:
    """Split 'cal_eval' correctly — dataset names like 'arc_challenge' contain underscores."""
    for cal in DATASETS:
        if combo_key.startswith(cal + "_"):
            return cal, combo_key[len(cal) + 1:]
    raise ValueError(f"Cannot parse combo_key: {combo_key!r}")


print(f"TF runs: {sorted(TF_URIS)}")

TF runs: ['arc_challenge_arc_challenge', 'arc_challenge_gsm8k', 'arc_challenge_humaneval', 'gsm8k_arc_challenge', 'gsm8k_gsm8k', 'gsm8k_humaneval', 'humaneval_arc_challenge', 'humaneval_gsm8k', 'humaneval_humaneval']


## Install scikit-learn if needed

scikit-learn is not in the project's core dependencies — install it if missing.

In [3]:
import subprocess

try:
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    print("scikit-learn already installed.")
except ImportError:
    print("Installing scikit-learn ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn"], check=True)
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    print("scikit-learn installed.")

Installing scikit-learn ...


  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.9 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 18.9 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


scikit-learn installed.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pruning_metrics.metrics import compute_kld, compute_jsd, compute_emd, compute_chamfer

METRIC_NAMES = ["kld", "jsd", "emd", "chamfer"]
_METRIC_FNS  = {
    "kld":     compute_kld,
    "jsd":     compute_jsd,
    "emd":     compute_emd,
    "chamfer": compute_chamfer,
}

# Calibration source colours — consistent with notebook 04
CAL_COLORS = {
    "arc_challenge": "tab:blue",
    "gsm8k":         "tab:orange",
    "humaneval":     "tab:green",
}

print("Imports OK.")

Imports OK.


## Build per-token file index

Scans the local TF cache (downloaded in notebook 04) and builds
`tf_index[combo_key][task_id][level_int] → Path`.
The `task_id` is read from inside the JSON to handle `_safe_filename` substitutions in directory names.

In [5]:
def _scan_combo_cache(combo_key: str) -> dict:
    """Returns {task_id: {level_int: Path}} for one TF run."""
    base = TF_CACHE_DIR / combo_key
    index: dict[str, dict[int, Path]] = {}
    if not base.exists():
        print(f"  WARNING: {base} not found — run notebook 04 first.")
        return index
    for level_dir in sorted(base.glob("level=*")):
        m = re.match(r"level=(\d+)", level_dir.name)
        if not m:
            continue
        level = int(m.group(1))
        for sample_dir in sorted(level_dir.iterdir()):
            pt_path = sample_dir / "per_token.json"
            if not pt_path.exists():
                continue
            with pt_path.open() as f:
                meta = json.load(f)
            tid = meta.get("task_id", sample_dir.name)
            index.setdefault(tid, {})[level] = pt_path
    return index


tf_index: dict[str, dict] = {}
for combo_key in TF_URIS:
    tf_index[combo_key] = _scan_combo_cache(combo_key)
    n_tasks = len(tf_index[combo_key])
    levels_found = sorted({lv for lv_dict in tf_index[combo_key].values() for lv in lv_dict})
    print(f"  {combo_key}: {n_tasks} tasks, levels={levels_found}")

  gsm8k_gsm8k: 1319 tasks, levels=[0, 20, 40, 60, 80]
  gsm8k_humaneval: 33 tasks, levels=[0, 20, 40, 60, 80]
  gsm8k_arc_challenge: 1171 tasks, levels=[0, 20, 40, 60, 80]
  humaneval_gsm8k: 1319 tasks, levels=[0, 20, 40, 60, 80]
  humaneval_humaneval: 33 tasks, levels=[0, 20, 40, 60, 80]
  humaneval_arc_challenge: 1171 tasks, levels=[0, 20, 40, 60, 80]
  arc_challenge_gsm8k: 1319 tasks, levels=[0, 20, 40, 60, 80]
  arc_challenge_humaneval: 33 tasks, levels=[0, 20, 40, 60, 80]
  arc_challenge_arc_challenge: 1171 tasks, levels=[0, 20, 40, 60, 80]


## Build model feature matrices

For each (eval benchmark, metric) pair, build a `(13, N)` matrix:
- **Row 0** — baseline: all zeros (metric of distribution vs. itself = 0)
- **Rows 1–12** — pruned models: one row per (cal, pruning level), values = per-example metric

Matrices are cached as `.npy` files so this heavy computation runs only once (~5–10 min total).

In [6]:
def build_model_matrix(
    eval_bench: str,
    metric_name: str,
) -> tuple[np.ndarray, list[dict]]:
    """
    Returns:
        X       : ndarray of shape (13, N_tasks)
        row_meta: list of 13 dicts, each {"cal": str, "level": int}
    """
    cache_npy  = RESULTS_DIR / f"tsne_model_matrix_{eval_bench}_{metric_name}.npy"
    cache_meta = RESULTS_DIR / f"tsne_model_matrix_{eval_bench}_{metric_name}_meta.json"

    if cache_npy.exists() and cache_meta.exists():
        X        = np.load(cache_npy)
        row_meta = json.loads(cache_meta.read_text())
        print(f"  {eval_bench}/{metric_name}: loaded from cache, shape={X.shape}")
        return X, row_meta

    # Collect task_ids present in ALL 3 cal combos for this eval bench
    combo_keys = {cal: f"{cal}_{eval_bench}" for cal in DATASETS
                  if f"{cal}_{eval_bench}" in tf_index}
    task_id_sets = [set(tf_index[ck].keys()) for ck in combo_keys.values()]
    task_ids = sorted(set.intersection(*task_id_sets))
    N = len(task_ids)
    print(f"  {eval_bench}/{metric_name}: N={N} tasks in common across {len(combo_keys)} cals")

    metric_fn = _METRIC_FNS[metric_name]

    # Row 0: baseline (all zeros)
    row_meta: list[dict] = [{"cal": "baseline", "level": 0}]
    rows: list[np.ndarray] = [np.zeros(N)]

    # Rows 1–12: pruned models
    for cal in DATASETS:
        combo_key  = f"{cal}_{eval_bench}"
        if combo_key not in tf_index:
            continue
        task_index = tf_index[combo_key]

        # Pre-load level=0 tokens once per (cal) — reused across all 4 pruning levels
        tokens_base: dict[str, list] = {}
        for tid in task_ids:
            lv_dict = task_index.get(tid, {})
            if 0 in lv_dict:
                with lv_dict[0].open() as f:
                    tokens_base[tid] = json.load(f).get("per_token", [])

        for level in NONZERO_LEVELS:
            row_vals: list[float] = []
            for j, tid in enumerate(task_ids):
                t0      = tokens_base.get(tid, [])
                lv_dict = task_index.get(tid, {})
                if not t0 or level not in lv_dict:
                    row_vals.append(float("nan"))
                    continue
                with lv_dict[level].open() as f:
                    tokens_k = json.load(f).get("per_token", [])
                if not tokens_k:
                    row_vals.append(float("nan"))
                    continue
                row_vals.append(metric_fn(t0, tokens_k))
            rows.append(np.array(row_vals, dtype=np.float64))
            row_meta.append({"cal": cal, "level": level})
            print(f"    {cal}/L{level}: done")

    X = np.stack(rows)  # (13, N)
    np.save(cache_npy, X)
    cache_meta.write_text(json.dumps(row_meta, indent=2))
    print(f"  {eval_bench}/{metric_name}: saved cache, shape={X.shape}")
    return X, row_meta


# Build all 12 matrices (3 benches × 4 metrics)
# Takes ~5–10 min on first run; subsequent runs load from cache.
model_matrices: dict[str, dict[str, tuple[np.ndarray, list]]] = {}
for eval_bench in DATASETS:
    model_matrices[eval_bench] = {}
    print(f"\n=== {BENCHMARK_LABELS[eval_bench]} ===")
    for metric_name in METRIC_NAMES:
        X, row_meta = build_model_matrix(eval_bench, metric_name)
        model_matrices[eval_bench][metric_name] = (X, row_meta)


=== ARC-Challenge ===
  arc_challenge/kld: N=1171 tasks in common across 3 cals
    arc_challenge/L20: done
    arc_challenge/L40: done
    arc_challenge/L60: done
    arc_challenge/L80: done
    gsm8k/L20: done
    gsm8k/L40: done
    gsm8k/L60: done
    gsm8k/L80: done
    humaneval/L20: done
    humaneval/L40: done
    humaneval/L60: done
    humaneval/L80: done
  arc_challenge/kld: saved cache, shape=(13, 1171)
  arc_challenge/jsd: N=1171 tasks in common across 3 cals
    arc_challenge/L20: done
    arc_challenge/L40: done
    arc_challenge/L60: done
    arc_challenge/L80: done
    gsm8k/L20: done
    gsm8k/L40: done
    gsm8k/L60: done
    gsm8k/L80: done
    humaneval/L20: done
    humaneval/L40: done
    humaneval/L60: done
    humaneval/L80: done
  arc_challenge/jsd: saved cache, shape=(13, 1171)
  arc_challenge/emd: N=1171 tasks in common across 3 cals
    arc_challenge/L20: done
    arc_challenge/L40: done
    arc_challenge/L60: done
    arc_challenge/L80: done
    gsm8k/L20

## Run t-SNE

Each (eval benchmark, metric) pair yields one 2-D embedding of 13 model points.

**Preprocessing**: `StandardScaler` normalises each example's metric values across the 13 model rows
(column-wise), so examples with very different absolute magnitudes contribute equally to the geometry.

**Parameters**:
- `perplexity=4`: must be < 13 (the number of points); gives ~4 effective nearest neighbours
- `max_iter=2000`: more iterations help convergence with this few points
- `init="pca"`: deterministic PCA initialisation, faster convergence than random

In [ ]:
tsne_results: dict[str, dict[str, np.ndarray]] = {}

for eval_bench in DATASETS:
    tsne_results[eval_bench] = {}
    for metric_name in METRIC_NAMES:
        X, row_meta = model_matrices[eval_bench][metric_name]

        # Replace any NaN rows with zeros (should not occur with complete cache)
        X_clean = np.where(np.isnan(X), 0.0, X)

        X_scaled = StandardScaler().fit_transform(X_clean)

        tsne = TSNE(
            n_components=2,
            perplexity=4,
            max_iter=2000,
            init="pca",
            random_state=42,
            n_jobs=1,
        )
        emb = tsne.fit_transform(X_scaled)
        tsne_results[eval_bench][metric_name] = emb
        print(
            f"  {eval_bench}/{metric_name}: "
            f"kl_divergence_={tsne.kl_divergence_:.4f}"
        )

## Plots: 4 figures, one per metric

Each figure has 3 panels (one per eval benchmark).
- **Black star** = baseline (unpruned, level=0) — conspicuously distinct
- **Coloured circles** = pruned models; colour = calibration source
- Each point labelled with its pruning level (`L20`, `L40`, etc.)

In [ ]:
FIGURES_DIR = RESULTS_DIR / "tsne_figures"
FIGURES_DIR.mkdir(exist_ok=True)

METRIC_TITLES = {
    "kld":     "KLD (KL Divergence)",
    "jsd":     "JSD (Jensen–Shannon Divergence)",
    "emd":     "EMD (Earth Mover's Distance)",
    "chamfer": "Chamfer Distance",
}

for metric_name in METRIC_NAMES:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, eval_bench in zip(axes, DATASETS):
        emb      = tsne_results[eval_bench][metric_name]  # (13, 2)
        row_meta = model_matrices[eval_bench][metric_name][1]  # list of 13 dicts

        # ── Baseline (row 0) ──────────────────────────────────────────────────
        ax.scatter(
            emb[0, 0], emb[0, 1],
            c="black", marker="*", s=500,
            zorder=5, label="baseline (L0)",
        )
        ax.annotate(
            "base",
            (emb[0, 0], emb[0, 1]),
            xytext=(6, 6), textcoords="offset points",
            fontsize=9, fontweight="bold",
        )

        # ── Pruned models (rows 1–12) ─────────────────────────────────────────
        for i, meta in enumerate(row_meta[1:], start=1):
            color = CAL_COLORS[meta["cal"]]
            ax.scatter(
                emb[i, 0], emb[i, 1],
                c=color, s=140,
                zorder=3, edgecolors="k", linewidths=0.4,
            )
            ax.annotate(
                f"L{meta['level']}",
                (emb[i, 0], emb[i, 1]),
                xytext=(5, 4), textcoords="offset points",
                fontsize=7.5,
            )

        ax.set_title(BENCHMARK_LABELS[eval_bench], fontsize=11, fontweight="bold")
        ax.set_xlabel("t-SNE dim 1", fontsize=8)
        ax.set_ylabel("t-SNE dim 2", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    # ── Shared legend ─────────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(color=CAL_COLORS[cal], label=f"cal = {BENCHMARK_LABELS[cal]}")
        for cal in DATASETS
    ]
    legend_handles.append(
        plt.scatter([], [], c="black", marker="*", s=200, label="baseline (L0)")
    )
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=4,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.06),
        frameon=True,
    )
    fig.suptitle(
        f"t-SNE — {METRIC_TITLES[metric_name]}\n"
        "Each point = one model (calibration source × pruning level)  |  "
        "Feature = per-example metric values across benchmark",
        fontsize=11, fontweight="bold", y=1.03,
    )
    fig.tight_layout()

    out_path = FIGURES_DIR / f"tsne_{metric_name}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

## Combined summary figure (4 × 3)

All 12 panels in one figure for side-by-side comparison across metrics and benchmarks.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 23))

for row_idx, metric_name in enumerate(METRIC_NAMES):
    for col_idx, eval_bench in enumerate(DATASETS):
        ax       = axes[row_idx, col_idx]
        emb      = tsne_results[eval_bench][metric_name]
        row_meta = model_matrices[eval_bench][metric_name][1]

        # Baseline
        ax.scatter(emb[0, 0], emb[0, 1], c="black", marker="*", s=400, zorder=5)
        ax.annotate(
            "base", (emb[0, 0], emb[0, 1]),
            xytext=(5, 5), textcoords="offset points",
            fontsize=7.5, fontweight="bold",
        )

        # Pruned models
        for i, meta in enumerate(row_meta[1:], start=1):
            ax.scatter(
                emb[i, 0], emb[i, 1],
                c=CAL_COLORS[meta["cal"]], s=100,
                zorder=3, edgecolors="k", linewidths=0.3,
            )
            ax.annotate(
                f"L{meta['level']}",
                (emb[i, 0], emb[i, 1]),
                xytext=(4, 3), textcoords="offset points",
                fontsize=6.5,
            )

        if row_idx == 0:
            ax.set_title(BENCHMARK_LABELS[eval_bench], fontsize=10, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(metric_name.upper(), fontsize=9, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])

legend_handles = [
    mpatches.Patch(color=CAL_COLORS[cal], label=f"cal = {BENCHMARK_LABELS[cal]}")
    for cal in DATASETS
]
legend_handles.append(
    plt.scatter([], [], c="black", marker="*", s=200, label="baseline (L0)")
)
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    fontsize=9,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle(
    "t-SNE of pruned model variants — rows = metric, cols = eval benchmark",
    fontsize=13, fontweight="bold", y=1.01,
)
fig.tight_layout()

out_path = FIGURES_DIR / "tsne_summary_all.png"
fig.savefig(out_path, dpi=120, bbox_inches="tight")
print(f"Saved summary: {out_path}")
plt.show()

## UMAP

Same model-as-point design as the t-SNE section above, but using UMAP for dimensionality reduction.

**Parameters**:
- `n_neighbors=4`: neighbourhood size (< 13 points); controls local vs global structure trade-off
- `min_dist=0.3`: minimum distance between embedded points; looser packing than default (0.1)
- `metric="euclidean"`: distance in the standardised feature space

In [ ]:
try:
    import umap
    print("umap-learn already installed.")
except ImportError:
    print("Installing umap-learn ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "umap-learn"], check=True)
    import umap
    print("umap-learn installed.")

In [ ]:
umap_results: dict[str, dict[str, np.ndarray]] = {}

for eval_bench in DATASETS:
    umap_results[eval_bench] = {}
    for metric_name in METRIC_NAMES:
        X, row_meta = model_matrices[eval_bench][metric_name]
        X_clean  = np.where(np.isnan(X), 0.0, X)
        X_scaled = StandardScaler().fit_transform(X_clean)

        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=4,   # must be < n_points=13; ~4 effective neighbours
            min_dist=0.3,
            metric="euclidean",
            random_state=42,
        )
        emb = reducer.fit_transform(X_scaled)
        umap_results[eval_bench][metric_name] = emb
        print(f"  {eval_bench}/{metric_name}: done")

In [ ]:
UMAP_FIGURES_DIR = RESULTS_DIR / "umap_figures"
UMAP_FIGURES_DIR.mkdir(exist_ok=True)

for metric_name in METRIC_NAMES:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, eval_bench in zip(axes, DATASETS):
        emb      = umap_results[eval_bench][metric_name]
        row_meta = model_matrices[eval_bench][metric_name][1]

        ax.scatter(
            emb[0, 0], emb[0, 1],
            c="black", marker="*", s=500,
            zorder=5,
        )
        ax.annotate(
            "base", (emb[0, 0], emb[0, 1]),
            xytext=(6, 6), textcoords="offset points",
            fontsize=9, fontweight="bold",
        )
        for i, meta in enumerate(row_meta[1:], start=1):
            ax.scatter(
                emb[i, 0], emb[i, 1],
                c=CAL_COLORS[meta["cal"]], s=140,
                zorder=3, edgecolors="k", linewidths=0.4,
            )
            ax.annotate(
                f"L{meta['level']}", (emb[i, 0], emb[i, 1]),
                xytext=(5, 4), textcoords="offset points", fontsize=7.5,
            )

        ax.set_title(BENCHMARK_LABELS[eval_bench], fontsize=11, fontweight="bold")
        ax.set_xlabel("UMAP dim 1", fontsize=8)
        ax.set_ylabel("UMAP dim 2", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    legend_handles = [
        mpatches.Patch(color=CAL_COLORS[cal], label=f"cal = {BENCHMARK_LABELS[cal]}")
        for cal in DATASETS
    ]
    legend_handles.append(plt.scatter([], [], c="black", marker="*", s=200, label="baseline (L0)"))
    fig.legend(
        handles=legend_handles,
        loc="lower center", ncol=4, fontsize=9,
        bbox_to_anchor=(0.5, -0.06), frameon=True,
    )
    fig.suptitle(
        f"UMAP — {METRIC_TITLES[metric_name]}\n"
        "Each point = one model (calibration source × pruning level)  |  "
        "Feature = per-example metric values across benchmark",
        fontsize=11, fontweight="bold", y=1.03,
    )
    fig.tight_layout()
    out_path = UMAP_FIGURES_DIR / f"umap_{metric_name}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 23))

for row_idx, metric_name in enumerate(METRIC_NAMES):
    for col_idx, eval_bench in enumerate(DATASETS):
        ax       = axes[row_idx, col_idx]
        emb      = umap_results[eval_bench][metric_name]
        row_meta = model_matrices[eval_bench][metric_name][1]

        ax.scatter(emb[0, 0], emb[0, 1], c="black", marker="*", s=400, zorder=5)
        ax.annotate(
            "base", (emb[0, 0], emb[0, 1]),
            xytext=(5, 5), textcoords="offset points",
            fontsize=7.5, fontweight="bold",
        )
        for i, meta in enumerate(row_meta[1:], start=1):
            ax.scatter(
                emb[i, 0], emb[i, 1],
                c=CAL_COLORS[meta["cal"]], s=100,
                zorder=3, edgecolors="k", linewidths=0.3,
            )
            ax.annotate(
                f"L{meta['level']}", (emb[i, 0], emb[i, 1]),
                xytext=(4, 3), textcoords="offset points", fontsize=6.5,
            )
        if row_idx == 0:
            ax.set_title(BENCHMARK_LABELS[eval_bench], fontsize=10, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(metric_name.upper(), fontsize=9, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])

legend_handles = [
    mpatches.Patch(color=CAL_COLORS[cal], label=f"cal = {BENCHMARK_LABELS[cal]}")
    for cal in DATASETS
]
legend_handles.append(plt.scatter([], [], c="black", marker="*", s=200, label="baseline (L0)"))
fig.legend(handles=legend_handles, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(
    "UMAP of pruned model variants — rows = metric, cols = eval benchmark",
    fontsize=13, fontweight="bold", y=1.01,
)
fig.tight_layout()
out_path = UMAP_FIGURES_DIR / "umap_summary_all.png"
fig.savefig(out_path, dpi=120, bbox_inches="tight")
print(f"Saved summary: {out_path}")
plt.show()

In [ ]:
print("Outputs:")
for p in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {p.relative_to(NOTEBOOK_DIR)}  ({p.stat().st_size / 1024:.0f} KB)")
print()
print("Cache files:")
for p in sorted(RESULTS_DIR.glob("tsne_model_matrix_*.npy")):
    print(f"  {p.relative_to(NOTEBOOK_DIR)}  ({p.stat().st_size / 1024:.0f} KB)")